# Exercise 1 · Simple RAG — follow the input and the output
**ETH MAS 2026 · Module 1 · B2 — From LLMs to agents**

```
your question ──► ① Search API ──► ② chunks become the prompt ──► ③ one LLM call ──► answer with sources
```
Run the cells **one by one**. After every step, look at what went **in** and what came **out**.
No coding needed — reading the outputs is the exercise.

**Time:** ~25 min · alone or in a group, as you like

## 0 · Setup (run once)

1. **File → Save a copy in Drive** — otherwise your changes are lost.
2. Run the next three cells. The last one should print a short greeting from the model.

The course API key is already in the notebook (see the cell below). *Optional, the clean way:* store it under the **key icon** in the left sidebar (*Secrets*) as `UNIQUE_API_KEY` — the notebook uses that first.

> After a break, Colab may have reset: use **Runtime → Run all** (or run from the top again).

In [1]:
%pip install -q unique-sdk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import re
import unique_sdk

# --- API key -------------------------------------------------------------------
# COURSE KEY: only for this course, revoked after Module 1 (21 Nov 2026).
# Never put real keys into a notebook — in production, keys live in a secret store.
COURSE_API_KEY = "ukey_4NzGvG0FxuZipyMzJdj6c7ftIObvRWt95ZwG8FnEfGQ"

try:  # the clean way: Colab Secrets (key icon in the left sidebar), if you set one
    from google.colab import userdata
    API_KEY = userdata.get("UNIQUE_API_KEY") or COURSE_API_KEY
except Exception:
    API_KEY = COURSE_API_KEY

# --- Course environment (provided by the lecturers) ---
unique_sdk.api_base = "https://gateway.unique.app/public/chat"
unique_sdk.api_key = API_KEY
unique_sdk.app_id = "app_eyugp8qq242fzp1lle4xjqc1"

COMPANY_ID = "346686658780860534"
USER_ID = "399599116591567964"
SCOPE_IDS = ["scope_nnqbp55xfh6s25wx4mdndff0"]   # knowledge base: company annual reports
MODEL = "AZURE_GPT_51_2025_1113"

In [3]:
# Connection check: one plain LLM call
hello = unique_sdk.ChatCompletion.create(
    company_id=COMPANY_ID,
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Say hello to the ETH MAS class in one short sentence."},
    ],
)
print(hello.choices[0].message.content)

Hello ETH MAS class—wishing you an inspiring and successful journey through your program!


## The question

In [4]:
QUESTION = "Give me an overview of Siemens AG."

## ① Search — input: your question · output: chunks
Nothing is generated yet. The Search API only returns text passages with document and page.

In [5]:
def search(query: str, limit: int = 5) -> list:
    """Search the knowledge base and return the list of matching chunks."""
    result = unique_sdk.Search.create(
        user_id=USER_ID,
        company_id=COMPANY_ID,
        searchString=query,
        searchType="VECTOR",
        scopeIds=SCOPE_IDS,
        limit=limit,
    )
    return result.data


chunks = search(QUESTION, limit=5)
print(f"{len(chunks)} chunks found\n")
for i, c in enumerate(chunks, 1):
    body = re.sub(r"<\|.*?\|>.*?<\|/.*?\|>", "", c["text"]).strip()
    print(f"[{i}] {c.get('key')}  p. {c.get('startPage')}–{c.get('endPage')}")
    print("    " + body[:180].replace("\n", " ") + " …\n")

5 chunks found

[1] Siemens_2022.pdf  p. 5–5
    # 1. Organization of the Siemens Group and basis of presentation  Siemens is a technology group that is active in nearly all countries of the world, focusing on the areas of automa …

[2] Siemens_2021.pdf  p. 5–5
    <!-- PageBreak -->   Combined Management Report   ## 1. Organization of the Siemens Group and basis of presentation  Siemens is a technology group that is active in nearly all coun …

[3] Siemens_2023.pdf  p. 5–5
    # 1. Organization of the Siemens Group and basis of presentation  Siemens is a technology group that is active in nearly all countries of the world, focusing on the areas of automa …

[4] Siemens_2023.pdf  p. 35–35
    ## 8.5 Significant characteristics of the internal control and risk management system   # 9. Siemens AG  The Annual Financial Statements of Siemens AG have been prepared in accorda …

[5] Siemens_2022.pdf  p. 35–35
    ## 8.5 Significant characteristics of the internal control and risk management 

> **Look at the output:**
> - How many chunks came back, and from which reports and pages?
> - Would *you* be able to answer the question from these passages alone?

In [6]:
# The raw JSON of ONE chunk (text shortened)
raw = dict(chunks[0]); raw["text"] = raw["text"][:300] + " …"
print(json.dumps(raw, indent=2, ensure_ascii=False))

Error: Failed to create kernel session on nbctl: nbctl vsessions unavailable: request to http://localhost:8888/api/internal/vsessions failed, reason: 

## ② Build the prompt — input: question + chunks · output: the exact text the LLM receives
This is **everything** the model will know. It has no access to the search, the files or your screen.

In [7]:
def build_context(chunks: list) -> str:
    parts = []
    for i, c in enumerate(chunks, 1):
        source = f"{c.get('title') or c.get('key')}, p. {c.get('startPage')}–{c.get('endPage')}"
        parts.append(f"[{i}] ({source})\n{c['text']}")
    return "\n\n".join(parts)

SYSTEM_PROMPT = """You are an analyst. Answer the question using ONLY the context below.
Cite the sources you use as [1], [2], ...
If the context does not contain the answer, say "I could not find this in the knowledge base."
"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Context:\n{build_context(chunks)}\n\nQuestion: {QUESTION}"},
]
user_text = messages[1]["content"]
print("SYSTEM:\n" + messages[0]["content"])
print(f"USER ({len(user_text):,} characters):\n{user_text[:900]}\n   [...]\n{user_text[-200:]}")

Error: Failed to create kernel session on nbctl: nbctl vsessions unavailable: request to http://localhost:8888/api/internal/vsessions failed, reason: 

> **Look at the output:**
> - Find the instruction that tells the model to cite sources and to say when it cannot find the answer.
> - How long is the prompt? What does that mean for cost?

## ③ Generate — input: the prompt · output: the answer

In [8]:
response = unique_sdk.ChatCompletion.create(
    company_id=COMPANY_ID, model=MODEL, messages=messages, options={"temperature": 0.0},
)
print(response.choices[0].message.content)

Error: Failed to create kernel session on nbctl: nbctl vsessions unavailable: request to http://localhost:8888/api/internal/vsessions failed, reason: 

> **Look at the output:**
> - Check two citations [n] against the chunks from step ①. Do they match?

---
## Your turn
Use `rag_answer(question, limit)` — the three steps in one function. Write down what you observe; we discuss it afterwards.

In [9]:
def rag_answer(question: str, limit: int = 5) -> str:
    chunks = search(question, limit=limit)
    response = unique_sdk.ChatCompletion.create(
        company_id=COMPANY_ID, model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{build_context(chunks)}\n\nQuestion: {question}"},
        ],
        options={"temperature": 0.0},
    )
    return response.choices[0].message.content

Error: Failed to create kernel session on nbctl: nbctl vsessions unavailable: request to http://localhost:8888/api/internal/vsessions failed, reason: 

**1 · With vs. without context.** Ask the LLM the same question *without* the chunks. What changes?

In [10]:
# Hint: unique_sdk.ChatCompletion.create(company_id=COMPANY_ID, model=MODEL, messages=[{"role": "user", "content": QUESTION}])


Error: Failed to create kernel session on nbctl: nbctl vsessions unavailable: request to http://localhost:8888/api/internal/vsessions failed, reason: 

**2 · How much context?** Compare `rag_answer(QUESTION, limit=1)` and `limit=10`.

Error: Failed to create kernel session on nbctl: nbctl vsessions unavailable: request to http://localhost:8888/api/internal/vsessions failed, reason: 

**3 · Not in the knowledge base.** Ask *"What is today's share price of Nestlé?"* — does the model admit it?

Error: Failed to create kernel session on nbctl: nbctl vsessions unavailable: request to http://localhost:8888/api/internal/vsessions failed, reason: 

**4 · A question with a hop.**
*"In fiscal 2021 Siemens completed a large acquisition. Which company was it, what did Siemens pay, and how did that business develop in fiscal 2022 and 2023?"*
Try `limit=5` and `limit=20` and look at the chunks: is there **any** passage from the 2023 report?

In [13]:
HOP_QUESTION = ("In fiscal 2021 Siemens completed a large acquisition. Which company was it, "
                "what did Siemens pay, and how did that business develop in fiscal 2022 and 2023?")


Error: Failed to create kernel session on nbctl: nbctl vsessions unavailable: request to http://localhost:8888/api/internal/vsessions failed, reason: 

## Take-away
- RAG = **retrieve → augment → generate**. The LLM only knows what is in the prompt.
- **One fixed search** is the weak spot: a second search that depends on what the first one found is impossible.
- ➜ In **Exercise 2** the LLM decides itself **what** to search and **how often** — that is an agent.